# Rules & systems summary

Point this at a scenario config and it tells you what the world *is*: every
transform, what each resource costs to simply keep existing, which way goods
flow along tagged roads, where priority changes come from, and — after a replay
— which rules actually fire versus which are dead letters.

The centrepiece is an interactive flow DAG with two views over the same layout:

| view | nodes | reads as |
|---|---|---|
| **Transforms** | resources + transforms | exact Petri net — nothing is lost |
| **Resource flow** | resources only | flow diagram; responsible transforms live in the edge tooltip |

**Hover any edge or node.** Everything else dims and a panel names the
transform(s) responsible, the stoichiometry, the net effect, and how much
actually flowed. Hovering a transform lights up its whole recipe.

`analysis.py` derives the facts (pure, no rendering); `dagviz.py` emits
self-contained SVG. Neither touches `sim/`.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root, so `import sim` works

from IPython.display import Markdown, display

from sim import load_scenario, initial_world, run_turn, SpendNudges
import analysis
import dagviz
import viz

DATA = Path.cwd().parent / "scenarios_data"

# ring-valley exercises the whole design: tagged roads with counter-flow, a
# control terminal minting nudges, and a cultural movement that reorders a
# stack on its own. simple-world is the minimal reference (and still collapses
# — see issue #1).
CONFIG = DATA / "ring-valley.json"

## 1. Load the config

Raw JSON alongside the compiled `Scenario` — the compiler drops `description`,
and it is worth showing.

In [ ]:
raw = json.loads(CONFIG.read_text())
scenario = load_scenario(CONFIG)

print(f"{scenario.name}: {scenario.R} resources, {scenario.T} transforms, "
      f"{scenario.L} locations")
print(f"evaluation order: {' -> '.join(raw['evaluation_order'])}")
print(f"edge tags: {', '.join(sorted(t for t in scenario.upstream_by_tag if t != 'nearby'))}")

## 2. Rules & systems

Read the **cheapest hold** column first — under universal decay it is the price of
merely continuing to exist, and it is where scenarios quietly go wrong. The
logistics and politics sections are derived from the config too: which tag moves
what, and which transforms can change a priority. Warnings at the bottom are
derived, not hand-written.

In [ ]:
display(Markdown(analysis.rules_summary(scenario, raw.get("description", ""))))

## 3. Replay

Run the engine forward and record what fired. This is what separates *rules that
exist* from *rules that matter* — and it drives the edge widths in the DAG below.

In [ ]:
runtime = analysis.run_and_observe(scenario, ticks=20)
display(Markdown(analysis.observed_summary(scenario, runtime)))

## 4. The flow DAG

Edge width is the volume observed over the replay; dashed coloured edges never
fired. **Dashed neutral edges are priority nudges** — a transform reordering
another transform, which is not a resource flow at all. Toggle **Resource flow**
for the collapsed view, where a self-loop is a storage path.

In [ ]:
dagviz.show(scenario, runtime=runtime)

## 5. Location topology

Which location may draw from which pool. Node labels are the initial stock. Note
this shows the raw graph — the *direction* goods actually travel comes from the
edge tags, not from this shape.

In [ ]:
world = initial_world(scenario, seed=0)
viz.draw_world(world, scenario, title="locations at tick 0");

## 6. Transport is just a transform

`haul_cityward` pulls wood one hop along the `cityward` tag each turn;
`haul_food_forestward` pulls food the other way along the *same roads*, because
the reverse edges carry the opposite tag. Neither is a special mechanic — both
are ordinary transforms whose input set happens to be a direction.

In [ ]:
w = initial_world(scenario, seed=0)
wood, food = scenario.resource_index["wood"], scenario.resource_index["food"]
print("        " + "".join(f"{lid:>22}" for lid in scenario.location_ids))
for t in range(6):
    cells = "".join(
        f"{f'wood {int(w.stock[l, wood])}, food {int(w.stock[l, food])}':>22}"
        for l in range(scenario.L))
    print(f"tick {t}  {cells}")
    w, _ = run_turn(w, scenario)

## 7. The world changes its own mind

Nobody submits an order below. The forest preaches a `back_to_the_land`
movement, it ships downstream one hop per turn, and `agitate` spends it to nudge
`farming` up and `logging` down — deltas that superpose and persist. Watch the
town's stack reorder itself.

In [ ]:
w = initial_world(scenario, seed=0)
town = scenario.location_index["town"]
farming = scenario.transform_names.index("farming")
logging_ = scenario.transform_names.index("logging")

print("tick   farming   logging   top of town's stack")
for t in range(9):
    print(f"{t:<6} {int(w.priority[town, farming]):+7d} {int(w.priority[town, logging_]):+9d}"
          f"   {scenario.transform_names[w.order(town)[0]]}")
    w, _ = run_turn(w, scenario)

## 8. Spending a nudge yourself

Your turn is as large as the capacity you built: `staff_control_terminal` mints
one `nudge` per turn, and each point of priority movement costs one. Unspent
nudges decay like any other resource, so political capital is use-it-or-lose-it.

In [ ]:
w = initial_world(scenario, seed=0)
for _ in range(3):
    w, _ = run_turn(w, scenario)

nudge = scenario.resource_index["nudge"]
ws = scenario.transform_names.index("wood_storage")
print(f"town holds {int(w.stock[town, nudge])} nudge(s), "
      f"wood_storage priority {int(w.priority[town, ws]):+d}")

after, _ = run_turn(w, scenario, orders=[SpendNudges("town", {"wood_storage": +1})])
print(f"after spending 1: wood_storage priority {int(after.priority[town, ws]):+d}")

try:
    run_turn(w, scenario, orders=[SpendNudges("town", {"wood_storage": +5})])
except ValueError as e:
    print(f"overspending is rejected: {e}")

## 9. The minimal scenario, for contrast

`simple-world` has no tags, no terminals and no actions — every transform draws
from `local` + `nearby`, which is exactly the pre-tag behaviour. It also still
collapses (issue #1), which the summary's warnings and the observed table make
visible.

In [ ]:
simple = load_scenario(DATA / "simple-world.json")
simple_runtime = analysis.run_and_observe(simple, ticks=8)
display(Markdown(analysis.observed_summary(simple, simple_runtime)))
dagviz.show(simple, runtime=simple_runtime, title="simple-world")